In [1]:
# ============================================================
# NAIVE BASELINE MODEL
# VERSION: USA + Canada
# ============================================================

import pandas as pd
import numpy as np
import ast

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("data_jobs.csv")

print("Original Shape:", df.shape)

Original Shape: (785741, 17)


In [3]:
# ============================================================
# KEEP ONLY UNITED STATES + CANADA
# ============================================================

valid_countries = ["United States", "Canada"]

df = df[df["job_country"].isin(valid_countries)].copy()

print("After Country Filter:", df.shape)

After Country Filter: (222321, 17)


In [4]:
# ============================================================
# CREATE UNIFIED ANNUAL SALARY
# ============================================================

# Use yearly salary if available
df["salary"] = df["salary_year_avg"]

# Convert hourly salary to annual salary
hourly_mask = (
    df["salary"].isna() &
    df["salary_hour_avg"].notna()
)

df.loc[hourly_mask, "salary"] = (
    df.loc[hourly_mask, "salary_hour_avg"] * 40 * 52
)

In [5]:
# ============================================================
# REMOVE ROWS WITH MISSING SALARY
# ============================================================

df = df[df["salary"].notna()].copy()

print("After Removing Missing Salaries:", df.shape)

After Removing Missing Salaries: (25685, 18)


In [6]:
# ============================================================
# REMOVE SALARY OUTLIERS USING IQR
# ============================================================

Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df = df[
    (df["salary"] >= lower_bound) &
    (df["salary"] <= upper_bound)
].copy()

print("After Removing Outliers:", df.shape)

After Removing Outliers: (25176, 18)


In [7]:
# ============================================================
# LOG TRANSFORM TARGET
# ============================================================

df["salary_log"] = np.log1p(df["salary"])

In [8]:
# ============================================================
# CLEAN job_skills COLUMN
# ============================================================

def clean_skills(skills):

    if pd.isna(skills):
        return []

    # If stored as string
    if isinstance(skills, str):

        try:
            parsed = ast.literal_eval(skills)

            if isinstance(parsed, list):
                skills = parsed

            else:
                skills = skills.split(",")

        except:
            skills = skills.split(",")

    if not isinstance(skills, list):
        return []

    cleaned = []

    for skill in skills:

        if pd.isna(skill):
            continue

        skill = str(skill).strip().lower()

        if skill != "":
            cleaned.append(skill)

    # Remove duplicates
    cleaned = list(dict.fromkeys(cleaned))

    return cleaned

df["clean_skills"] = df["job_skills"].apply(clean_skills)

In [9]:
# ============================================================
# CREATE SIMPLE SKILL COUNT FEATURE
# ============================================================

df["skill_count"] = df["clean_skills"].apply(len)

In [10]:
# ============================================================
# REMOVE RARE JOB TITLES
# Keep titles appearing >= 10 times
# ============================================================

title_counts = df["job_title_short"].value_counts()

valid_titles = title_counts[title_counts >= 10].index

df = df[df["job_title_short"].isin(valid_titles)].copy()

print("After Removing Rare Titles:", df.shape)

After Removing Rare Titles: (25176, 21)


In [11]:
# ============================================================
# SELECT SIMPLE BASELINE FEATURES
# ============================================================

baseline_features = [
    "job_title_short",
    "job_country",
    "job_schedule_type",
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance",
    "skill_count"
]

X = df[baseline_features].copy()
y = df["salary_log"]

In [12]:
# ============================================================
# CONVERT BOOLEAN COLUMNS TO INTEGERS
# ============================================================

binary_features = [
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance"
]

for col in binary_features:
    X[col] = X[col].astype("Int64")

In [13]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
# ============================================================
# FEATURE GROUPS
# ============================================================

categorical_features = [
    "job_title_short",
    "job_country",
    "job_schedule_type"
]

numeric_features = [
    "skill_count",
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance"
]

In [15]:
# ============================================================
# PREPROCESSING PIPELINE
# ============================================================

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    ]
)

In [16]:
# ============================================================
# BASELINE MODEL
# ============================================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

In [17]:
# ============================================================
# TRAIN MODEL
# ============================================================

print("\nTraining Baseline Model...\n")

model.fit(X_train, y_train)

print("Training Complete!")


Training Baseline Model...

Training Complete!


In [18]:
# ============================================================
# PREDICT
# ============================================================

pred_log = model.predict(X_test)

# Convert back to original salary scale
pred_salary = np.expm1(pred_log)
actual_salary = np.expm1(y_test)

In [19]:
# ============================================================
# EVALUATION
# ============================================================

r2 = r2_score(actual_salary, pred_salary)

mae = mean_absolute_error(
    actual_salary,
    pred_salary
)

rmse = np.sqrt(
    mean_squared_error(
        actual_salary,
        pred_salary
    )
)

In [20]:
# ============================================================
# RESULTS
# ============================================================

print("\n========== BASELINE RESULTS ==========")

print(f"R² Score : {r2:.4f}")
print(f"MAE      : ${mae:,.2f}")
print(f"RMSE     : ${rmse:,.2f}")


========== BASELINE RESULTS ==========
R² Score : 0.2540
MAE      : $28,765.69
RMSE     : $36,725.98


In [21]:
# ============================================================
# OPTIONAL: VIEW SAMPLE PREDICTIONS
# ============================================================

results = pd.DataFrame({
    "Actual Salary": actual_salary.values,
    "Predicted Salary": pred_salary
})

print("\nSample Predictions:")
print(results.head(10))


Sample Predictions:
   Actual Salary  Predicted Salary
0  122000.000000      84821.011385
1  110000.000000     132784.935761
2   73725.599365     100156.773147
3   85000.000000      84603.760785
4  150000.000000     133143.384485
5   53300.000000      82379.048137
6   88500.000000     124604.630966
7  164000.000000     121449.263192
8  125000.000000     119560.174088
9   31200.000000     144450.071778
